# PathSnap: shared preprocessing for six selected features
Run in Google Colab on CPU. This notebook builds inputs for **all five models**; it does not choose or train the models.

Order: `distance_m`, `distance_rank`, `heading_difference_deg`, `projection_fraction`, `segment_length_m`, `displacement_m`.

One row = one GPS observation–candidate directed road segment. Label 1 = candidate matches the route-derived reference segment; otherwise 0. IDs remain metadata, never features.

The original 80 training trips, 10 student-test trips, 75 m radius, maximum eight candidates, weak-label rules and exact saved five trip folds are retained. Evaluator data is not read.

**Benchmark limit:** the supplied routes have no per-observation truth. Distance and heading help derive weak labels and also appear in X, so scores can be optimistic. If feature selection used held-out data, use an independent test set for an unbiased final estimate.


In [1]:
%pip -q install 'numpy>=1.26,<3' 'pandas>=2.2,<4' 'scipy>=1.11,<2' 'scikit-learn>=1.4,<2' 'shapely>=2.0,<3' 'pyproj>=3.6,<4' 'joblib>=1.3,<2'


## 1. Dataset location
Choose `drive` for a folder already on Drive (edit `DATA_ROOT`), or `zip` to upload `Student folder.zip`.
Outputs are written to a separate Shared6 folder. CPU is sufficient.


In [2]:
from pathlib import Path
SOURCE = 'drive'  # 'drive' or 'zip'
if SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/PathSnap_ip')  # train/ and student_test/ inside
    OUTPUT_DIR = Path('/content/drive/MyDrive/output/Shared6')
elif SOURCE == 'zip':
    from google.colab import files
    import zipfile
    ZIP_PATH = Path('/content/Student folder.zip')
    if not ZIP_PATH.exists():
        files.upload()  # Select Student folder.zip
    DATA_ROOT = Path('/content/pathsnap_raw')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for member in archive.infolist():
            target = (DATA_ROOT / member.filename).resolve()
            if not target.is_relative_to(DATA_ROOT.resolve()):
                raise ValueError('Archive contains a path outside the extraction folder')
        archive.extractall(DATA_ROOT)
    OUTPUT_DIR = Path('/content/Shared6')
else:
    raise ValueError("SOURCE must be 'drive' or 'zip'")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Raw data:', DATA_ROOT)
print('Shared output:', OUTPUT_DIR)


Mounted at /content/drive
Raw data: /content/drive/MyDrive/PathSnap_ip
Shared output: /content/drive/MyDrive/output/Shared6


## 2. Shared preprocessing code
Invalid GPS rows and duplicate timestamps are removed before feature generation; timestamps are sorted within each trip. Coordinates use local UTM metres. Heading is 90° when movement is under 3 m or the preceding time gap exceeds 30 s; displacement is zero for the first observation. Projection fraction is clipped to [0,1]. Missing candidate observations are retained in observation ledgers for evaluation.


In [3]:
# %% Imports and fixed experimental protocol
from dataclasses import asdict, dataclass
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import argparse
import importlib.metadata
import json
import os
import platform

import numpy as np
import pandas as pd
import shapely
from pyproj import Geod, Transformer
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from shapely import STRtree
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             log_loss, precision_score, recall_score)
from sklearn.model_selection import GroupKFold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


@dataclass(frozen=True)
class Protocol:
    schema_version: str = "shared-six-v1"
    candidate_radius_m: float = 75.0
    max_candidates: int = 8
    heading_min_displacement_m: float = 3.0
    heading_max_gap_s: float = 30.0
    n_splits: int = 5
    random_state: int = RANDOM_STATE


# Exactly six ordered numeric model inputs. IDs and labels stay outside X.
# First observations have displacement=0 and heading_difference_deg=90.
# No heading-validity indicator is included in this selected feature set.
FEATURE_INFO = [('distance_m', 'Shortest projected GPS-to-segment distance (metres).', 'Measures spatial compatibility.'), ('distance_rank', '1-based distance rank; equal distances break by arc ID.', 'Expresses relative proximity among candidates.'), ('heading_difference_deg', 'Absolute wrapped movement/segment heading difference, 0..180; 90 if unavailable.', 'Distinguishes opposite travel directions.'), ('projection_fraction', 'Clipped fraction along segment of the perpendicular GPS projection, 0..1.', 'Identifies endpoint and interior projections.'), ('segment_length_m', 'Length of the directed straight segment (metres).', 'Distinguishes short connectors from long roads.'), ('displacement_m', 'Projected displacement from the preceding retained GPS observation; first=0.', 'Describes movement magnitude and GPS noise.')]
FEATURES = [f[0] for f in FEATURE_INFO]
METRICS = ["Accuracy", "Precision", "Recall", "F1", "Log Loss", "Top-1 Accuracy",
           "Candidate Miss Rate", "Train Time", "Inference Time"]
COMPARISON_COLUMNS = ["Model"] + METRICS


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")


def resolve_data_root(path):
    """Accept either input/, input/Student folder/, or an equivalent Colab path."""
    path = Path(path).expanduser().resolve()
    for root in (path, path / "Student folder"):
        if (root / "train").is_dir() and (root / "student_test").is_dir():
            return root
    raise FileNotFoundError(f"Expected train/ and student_test/ under {path} or its Student folder/ child")


def discover_trips(root):
    """Allowlist exactly two splits; never open or enumerate evaluator files."""
    splits = {}
    for split, expected in (("train", 80), ("student_test", 10)):
        trips = sorted(p for p in (root / split).iterdir() if p.is_dir())
        if len(trips) != expected:
            raise ValueError(f"{split}: expected {expected} trips, found {len(trips)}")
        for trip in trips:
            for ext in ("track", "nodes", "arcs", "route"):
                path = trip / f"{trip.name}.{ext}"
                if not path.is_file() or path.stat().st_size == 0:
                    raise FileNotFoundError(f"Required nonempty supervised input missing: {path}")
        splits[split] = trips
    if {p.name for p in splits["train"]} & {p.name for p in splits["student_test"]}:
        raise ValueError("Trip IDs overlap between train and student_test")
    return splits


def read_numeric(path, columns, integer=False):
    """Parse headerless whitespace, reject wrong columns/non-numeric/invalid IDs."""
    try:
        a = np.loadtxt(path, dtype=np.float64, ndmin=2)
    except (ValueError, OSError) as exc:
        raise ValueError(f"Cannot parse {path}: {exc}") from exc
    if not len(a) or a.shape[1] != columns:
        raise ValueError(f"{path}: expected {columns} columns, got shape {a.shape}")
    if integer:
        if not np.isfinite(a).all() or not np.equal(a, np.floor(a)).all():
            raise ValueError(f"{path}: expected finite integer IDs")
        a = a.astype(np.int64)
    return a


def coordinates_valid(a):
    return np.isfinite(a[:, :2]).all(axis=1) & (np.abs(a[:, 0]) <= 180) & (np.abs(a[:, 1]) <= 90)


def preprocess_track(raw):
    """Stable timestamp sort, retain original row IDs, drop invalid/duplicate time rows."""
    good = coordinates_valid(raw) & np.isfinite(raw[:, 2])
    frame = pd.DataFrame(raw[good], columns=["longitude", "latitude", "timestamp"])
    frame["raw_row"] = np.flatnonzero(good)
    original_order = frame.raw_row.to_numpy().copy()
    frame = frame.sort_values("timestamp", kind="stable")
    reordered = int(np.sum(original_order != frame.raw_row.to_numpy()))
    duplicates = int(frame.timestamp.duplicated().sum())
    frame = frame.drop_duplicates("timestamp", keep="first").reset_index(drop=True)
    if len(frame) < 2:
        raise ValueError("Trip has fewer than two valid, distinct-time GPS observations")
    assert (np.diff(frame.timestamp) > 0).all()
    return frame, dict(raw_observations=len(raw), invalid_gps_dropped=int((~good).sum()),
                       duplicate_times_dropped=duplicates, reordered_rows=reordered)


@dataclass
class Network:
    nodes_xy: np.ndarray
    arcs: np.ndarray
    lines: np.ndarray
    lengths: np.ndarray
    headings: np.ndarray
    active_ids: np.ndarray
    tree: STRtree
    out_degree: np.ndarray
    in_degree: np.ndarray


def make_network(nodes_xy, arcs):
    endpoints = nodes_xy[arcs]
    delta = endpoints[:, 1] - endpoints[:, 0]
    lengths = np.linalg.norm(delta, axis=1)
    active = np.flatnonzero(lengths > 1e-6)
    if not len(active):
        raise ValueError("No nondegenerate road segments")
    lines = shapely.linestrings(endpoints)
    return Network(nodes_xy, arcs, lines, lengths, np.arctan2(delta[:, 1], delta[:, 0]),
                   active, STRtree(lines[active]),
                   np.bincount(arcs[active, 0], minlength=len(nodes_xy)),
                   np.bincount(arcs[active, 1], minlength=len(nodes_xy)))


def load_trip(trip):
    paths = {ext: trip / f"{trip.name}.{ext}" for ext in ("track", "nodes", "arcs", "route")}
    track, quality = preprocess_track(read_numeric(paths["track"], 3))
    nodes = read_numeric(paths["nodes"], 2)
    arcs = read_numeric(paths["arcs"], 2, integer=True)
    route = read_numeric(paths["route"], 1, integer=True).ravel()
    if not coordinates_valid(nodes).all():
        raise ValueError(f"{trip}: nonfinite/out-of-range node coordinates")
    if arcs.min() < 0 or arcs.max() >= len(nodes):
        raise ValueError(f"{trip}: arc endpoint outside zero-based node index")
    if route.min() < 0 or route.max() >= len(arcs):
        raise ValueError(f"{trip}: route ID outside zero-based arc index")
    # A separate local UTM zone/hemisphere for each trip. No route labels or
    # fitted statistics are used by this deterministic coordinate conversion.
    lon, lat = track[["longitude", "latitude"]].median().to_numpy()
    if not -80 <= lat <= 84:
        raise ValueError(f"{trip}: outside supported UTM latitude range")
    zone = min(60, max(1, int((lon + 180) // 6) + 1))
    epsg = (32600 if lat >= 0 else 32700) + zone
    transform = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    nodes_xy = np.column_stack(transform.transform(nodes[:, 0], nodes[:, 1]))
    track_xy = np.column_stack(transform.transform(track.longitude.to_numpy(), track.latitude.to_numpy()))
    if not np.isfinite(nodes_xy).all() or not np.isfinite(track_xy).all():
        raise ValueError(f"{trip}: invalid projection output")
    net = make_network(nodes_xy, arcs)
    adj = coo_matrix((np.ones(len(arcs), dtype=np.int8), (arcs[:, 0], arcs[:, 1])),
                     shape=(len(nodes), len(nodes))).tocsr()
    components = connected_components(adj, directed=False, return_labels=False)
    sample = net.active_ids[np.linspace(0, len(net.active_ids) - 1, min(200, len(net.active_ids)), dtype=int)]
    a, b = nodes[arcs[sample, 0]], nodes[arcs[sample, 1]]
    _, _, geod_lengths = Geod(ellps="WGS84").inv(a[:, 0], a[:, 1], b[:, 0], b[:, 1])
    relative_errors = np.abs(net.lengths[sample] / geod_lengths - 1)
    if relative_errors.max() > 0.01:
        raise ValueError(f"{trip}: sampled UTM/geodesic length discrepancy exceeds 1%; review CRS")
    dt = np.diff(track.timestamp)
    quality.update(trip_id=trip.name, observations=len(track), nodes=len(nodes), arcs=len(arcs),
                   route_entries=len(route), route_repeated_arcs=int(len(route) - len(np.unique(route))),
                   route_disconnects=int((arcs[route[:-1], 1] != arcs[route[1:], 0]).sum()),
                   zero_length_arcs=int((net.lengths <= 1e-6).sum()),
                   weak_components=int(components), isolated_nodes=int((net.in_degree + net.out_degree == 0).sum()),
                   epsg=epsg, projection_max_relative_error=float(relative_errors.max()),
                   longitude_min=float(track.longitude.min()), longitude_max=float(track.longitude.max()),
                   latitude_min=float(track.latitude.min()), latitude_max=float(track.latitude.max()),
                   timestamp_min=float(track.timestamp.min()), timestamp_max=float(track.timestamp.max()),
                   dt_min=float(dt.min()), dt_max=float(dt.max()), steps_1s=int((dt == 1).sum()),
                   gaps_over_30s=int((dt > 30).sum()))
    return track, track_xy, net, route, quality


# %% Label-free spatial candidates and common features
def movement_features(track, xy, protocol):
    delta = np.diff(xy, axis=0, prepend=xy[:1])
    displacement = np.linalg.norm(delta, axis=1)
    dt = np.diff(track.timestamp, prepend=track.timestamp.iloc[0])
    valid = (displacement >= protocol.heading_min_displacement_m) & (dt > 0) & (dt <= protocol.heading_max_gap_s)
    speed = np.divide(displacement, dt, out=np.zeros_like(dt), where=dt > 0)
    return dt, displacement, speed, np.arctan2(delta[:, 1], delta[:, 0]), valid


def heading_difference(movement, segment):
    return np.rad2deg(np.abs((movement - segment + np.pi) % (2 * np.pi) - np.pi))


def generate_candidates(track, xy, net, protocol=Protocol()):
    """R-tree radius search, exact distances, nearest k. Never receives .route.

    No forced reference injection and no label-dependent radius expansion.
    Observations without a segment within the radius remain in the ledger.
    """
    points = shapely.points(xy)
    pair = net.tree.query(points, predicate="dwithin", distance=protocol.candidate_radius_m)
    obs_id = pair[0].astype(np.int32)
    arc_id = net.active_ids[pair[1]].astype(np.int32)
    distances = shapely.distance(points[obs_id], net.lines[arc_id])
    # Stable deterministic ordering, including reversed arcs with equal geometry.
    order = np.lexsort((arc_id, distances, obs_id))
    obs_id, arc_id, distances = obs_id[order], arc_id[order], distances[order]
    starts = np.r_[0, np.flatnonzero(np.diff(obs_id)) + 1] if len(obs_id) else np.array([], dtype=int)
    rank = np.arange(len(obs_id)) - np.repeat(starts, np.diff(np.r_[starts, len(obs_id)])) + 1
    keep = rank <= protocol.max_candidates
    obs_id, arc_id, distances, rank = obs_id[keep], arc_id[keep], distances[keep], rank[keep]
    dt, displacement, speed, heading, valid = movement_features(track, xy, protocol)
    endpoints = net.nodes_xy[net.arcs[arc_id]]
    vector = endpoints[:, 1] - endpoints[:, 0]
    fraction = np.einsum("ij,ij->i", xy[obs_id] - endpoints[:, 0], vector) / net.lengths[arc_id] ** 2
    hd = heading_difference(heading[obs_id], net.headings[arc_id])
    hd[~valid[obs_id]] = 90.0
    values = np.column_stack([
        distances, rank, hd, np.clip(fraction, 0, 1),
        net.lengths[arc_id], displacement[obs_id],
    ]).astype(np.float32)
    candidates = pd.DataFrame(values, columns=FEATURES)
    candidates.insert(0, "arc_id", arc_id)
    candidates.insert(0, "observation_id", obs_id)
    assert np.isfinite(values).all()
    assert not candidates.duplicated(["observation_id", "arc_id"]).any()
    return candidates


# %% Independent construction of geometric reference labels
def derive_reference_labels(track, xy, net, route, protocol=Protocol()):
    """Weak observation labels from the supplied route, independently of candidates.

    Choose the nearest nonzero directed segment anywhere on the reference route.
    Exact geometric ties: heading agreement (if valid), then proximity to a
    rough progress estimate from cumulative observed displacement, then route
    position. Chronology informs ties only: this is not a sequence decoder.
    Route breaks are recorded, never filled with invented connecting segments.
    """
    positions = np.flatnonzero(net.lengths[route] > 1e-6)
    if not len(positions):
        raise ValueError("Reference route has no nondegenerate segment")
    ids = route[positions]
    points = shapely.points(xy)
    pairs, distance = STRtree(net.lines[ids]).query_nearest(points, all_matches=True, return_distance=True)
    obs_id, route_pos = pairs[0], positions[pairs[1]]
    arc_id = route[route_pos]
    _, displacement, _, heading, valid = movement_features(track, xy, protocol)
    prefix = np.r_[0.0, np.cumsum(net.lengths[route])]
    observed_progress = np.cumsum(displacement)
    expected_progress = observed_progress / max(observed_progress[-1], 1e-6) * prefix[-1]
    projected = shapely.line_locate_point(net.lines[arc_id], points[obs_id])
    progress = prefix[route_pos] + projected
    hd = heading_difference(heading[obs_id], net.headings[arc_id])
    hd[~valid[obs_id]] = 0
    order = np.lexsort((route_pos, np.abs(progress - expected_progress[obs_id]), hd, obs_id))
    ordered_obs = obs_id[order]
    selected = order[np.r_[True, np.diff(ordered_obs) != 0]]
    assert np.array_equal(obs_id[selected], np.arange(len(track)))
    matches = pd.DataFrame({"obs": obs_id, "arc": arc_id}).drop_duplicates()
    ambiguous = np.bincount(matches.obs.to_numpy(), minlength=len(track)) > 1
    return pd.DataFrame(dict(
        reference_arc_id=arc_id[selected].astype(np.int32),
        reference_distance_m=distance[selected], reference_route_position=route_pos[selected],
        reference_progress_m=progress[selected], reference_tie=ambiguous,
    ))


def attach_labels(candidates, observations):
    """Keep candidate misses, including observations with zero candidate rows."""
    candidates = candidates.copy()
    obs_idx = candidates.observation_id.to_numpy()
    candidates["label"] = (candidates.arc_id.to_numpy() == observations.reference_arc_id.to_numpy()[obs_idx]).astype(np.int8)
    counts = np.bincount(obs_idx, minlength=len(observations))
    positives = np.bincount(obs_idx, weights=candidates.label.to_numpy(), minlength=len(observations))
    observations = observations.copy()
    observations["candidate_count"] = counts
    observations["reference_present"] = positives > 0
    assert (positives <= 1).all()
    assert int(candidates.label.sum()) == int(observations.reference_present.sum())
    return candidates, observations


@dataclass
class Dataset:
    candidates: pd.DataFrame
    observations: pd.DataFrame


def input_manifest(splits):
    """Content hashes make caches invalid when any permitted source file changes."""
    files = []
    for split, trips in splits.items():
        for trip in trips:
            for ext in ("track", "nodes", "arcs", "route"):
                path = trip / f"{trip.name}.{ext}"
                files.append(dict(split=split, trip_id=trip.name, file=path.name,
                                  bytes=path.stat().st_size, sha256=sha256(path.read_bytes()).hexdigest()))
    return files


def build_dataset(trips, split, protocol, out):
    candidate_frames, observation_frames, inventory = [], [], []
    offset = 0
    for number, trip in enumerate(trips, 1):
        start = perf_counter()
        track, xy, net, route, quality = load_trip(trip)
        candidates = generate_candidates(track, xy, net, protocol)
        # Reference information enters here, after all features are finished.
        labels = derive_reference_labels(track, xy, net, route, protocol)
        observations = pd.concat([track, labels], axis=1)
        candidates, observations = attach_labels(candidates, observations)
        observations.insert(0, "observation_id", np.arange(len(track)) + offset)
        observations.insert(0, "trip_id", trip.name)
        candidates.observation_id += offset
        candidates.insert(0, "trip_id", trip.name)
        quality.update(split=split, candidate_rows=len(candidates),
                       positive_candidates=int(candidates.label.sum()),
                       candidate_misses=int((~observations.reference_present).sum()),
                       zero_candidate_observations=int((observations.candidate_count == 0).sum()),
                       reference_ties=int(observations.reference_tie.sum()),
                       reference_over_radius=int((observations.reference_distance_m > protocol.candidate_radius_m).sum()),
                       reference_progress_backsteps=int((np.diff(observations.reference_progress_m) < -1).sum()),
                       preprocessing_seconds=perf_counter() - start)
        inventory.append(quality)
        candidate_frames.append(candidates)
        observation_frames.append(observations)
        offset += len(track)
        if number == 1 or number % 10 == 0 or number == len(trips):
            print(f"{split}: prepared {number}/{len(trips)} trips ({offset:,} observations)", flush=True)
    dataset = Dataset(pd.concat(candidate_frames, ignore_index=True), pd.concat(observation_frames, ignore_index=True))
    save_dataset(dataset, out, split)
    pd.DataFrame(inventory).to_csv(out / f"{split}_inventory.csv", index=False)
    return dataset


def save_dataset(dataset, out, split):
    c = dataset.candidates
    np.savez_compressed(out / f"{split}_features.npz", X=c[FEATURES].to_numpy(np.float32),
                        label=c.label.to_numpy(np.int8), trip_id=c.trip_id.to_numpy(dtype="U"),
                        observation_id=c.observation_id.to_numpy(np.int32), arc_id=c.arc_id.to_numpy(np.int32))
    c.to_csv(out / f"{split}_candidates.csv.gz", index=False, compression="gzip")
    dataset.observations.to_csv(out / f"{split}_observations.csv.gz", index=False, compression="gzip")


def load_dataset(out, split):
    with np.load(out / f"{split}_features.npz", allow_pickle=False) as cache:
        c = pd.DataFrame(cache["X"], columns=FEATURES)
        for key in ("trip_id", "observation_id", "arc_id", "label"):
            c[key] = cache[key]
    obs = pd.read_csv(out / f"{split}_observations.csv.gz", dtype={"trip_id": str})
    return Dataset(c, obs)


def prepare_datasets(data_root, output_dir, protocol=Protocol(), rebuild=False):
    root = resolve_data_root(data_root)
    out = Path(output_dir).expanduser().resolve()
    if out == root or root in out.parents:
        raise ValueError("Output directory must be outside the raw dataset")
    out.mkdir(parents=True, exist_ok=True)
    splits = discover_trips(root)
    print("Verifying content hashes of the 360 permitted input files...", flush=True)
    manifest = input_manifest(splits)
    signature = {"protocol": asdict(protocol), "features": FEATURES, "inputs": manifest,
                 "spatial_versions": {k: importlib.metadata.version(k) for k in ("numpy", "shapely", "pyproj")}}
    # This fixed schema version must be incremented for changes to feature/label
    # semantics. Protocol values, file contents and spatial versions are hashed.
    digest = sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()
    cache_path = out / "cache_manifest.json"
    required = [out / f"{s}_{suffix}" for s in splits for suffix in
                ("features.npz", "observations.csv.gz", "inventory.csv", "candidates.csv.gz")]
    valid_cache = (not rebuild and cache_path.is_file() and all(p.is_file() for p in required)
                   and json.loads(cache_path.read_text(encoding="utf-8"))["digest"] == digest)
    if valid_cache:
        print("Reusing verified common feature/label caches.", flush=True)
        datasets = {s: load_dataset(out, s) for s in splits}
    else:
        datasets = {s: build_dataset(trips, s, protocol, out) for s, trips in splits.items()}
        write_json(cache_path, dict(digest=digest, **signature))
    write_json(out / "feature_schema.json", [dict(name=n, definition=d, rationale=r) for n, d, r in FEATURE_INFO])
    pd.concat([pd.read_csv(out / f"{s}_inventory.csv", dtype={"trip_id": str}) for s in splits]).to_csv(out / "dataset_inventory.csv", index=False)
    return datasets


# %% Shared model-independent metrics and trip-aware folds
def evaluate_predictions(dataset, probabilities, train_seconds=0.0, inference_seconds=0.0):
    """Binary metrics on generated rows; Top-1 on ALL labeled observations.

    Miss rate is a fraction (multiply by 100 for percent). Missing-reference and
    zero-candidate observations count as pipeline Top-1 failures. A separate
    covered-only score isolates the classifier from candidate-generation misses.
    All ranking ties prefer the existing distance-rank/arc-ID order.
    """
    c, obs = dataset.candidates, dataset.observations
    p = np.asarray(probabilities, dtype=np.float64)
    if p.shape != (len(c),) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Expected one finite probability in [0, 1] per candidate")
    if c.empty:
        raise ValueError("No candidate rows in evaluation partition; classification metrics undefined")
    y, pred = c.label.to_numpy(), (p >= 0.5).astype(np.int8)
    scored = c[["observation_id", "arc_id", "distance_rank"]].copy()
    scored["probability"] = p
    best = scored.sort_values(["observation_id", "probability", "distance_rank", "arc_id"],
                              ascending=[True, False, True, True], kind="stable").drop_duplicates("observation_id")
    per_obs = obs.merge(best[["observation_id", "arc_id", "probability"]], on="observation_id", how="left", validate="one_to_one")
    per_obs.rename(columns={"arc_id": "predicted_arc_id", "probability": "predicted_probability"}, inplace=True)
    per_obs["top1_correct"] = per_obs.predicted_arc_id == per_obs.reference_arc_id
    covered = per_obs.reference_present.astype(bool)
    matrix = confusion_matrix(y, pred, labels=[0, 1])
    scores = dict(zip(METRICS, [
        float(accuracy_score(y, pred)), float(precision_score(y, pred, zero_division=0)),
        float(recall_score(y, pred, zero_division=0)), float(f1_score(y, pred, zero_division=0)),
        float(log_loss(y, np.column_stack([1 - p, p]), labels=[0, 1])),
        float(per_obs.top1_correct.mean()), float((~covered).mean()),
        float(train_seconds), float(inference_seconds),
    ]))
    # The supplied final-report template also asks for Macro F1; keep the
    # requested common F1 explicitly positive-class, and supply macro separately.
    scores.update({"Macro F1": float(f1_score(y, pred, average="macro", labels=[0, 1], zero_division=0)),
                   "Top-1 Accuracy (covered)": float(per_obs.loc[covered, "top1_correct"].mean()) if covered.any() else None,
                   "Observations": len(obs), "Candidates": len(c),
                   "Candidate Miss Count": int((~covered).sum()),
                   "Zero Candidate Observations": int((per_obs.candidate_count == 0).sum())})
    return scores, matrix, per_obs


REFERENCE_FOLDS = {'00000001': 4, '00000002': 5, '00000003': 3, '00000004': 4, '00000005': 4, '00000006': 2, '00000008': 1, '00000010': 1, '00000011': 5, '00000012': 3, '00000013': 4, '00000014': 5, '00000016': 2, '00000018': 3, '00000019': 4, '00000020': 1, '00000021': 3, '00000022': 5, '00000024': 2, '00000025': 5, '00000026': 1, '00000027': 1, '00000028': 4, '00000029': 5, '00000031': 2, '00000032': 3, '00000033': 4, '00000034': 1, '00000035': 2, '00000036': 3, '00000037': 2, '00000038': 4, '00000039': 2, '00000041': 3, '00000042': 5, '00000043': 2, '00000044': 1, '00000045': 4, '00000046': 1, '00000047': 3, '00000048': 1, '00000050': 3, '00000051': 5, '00000053': 5, '00000054': 1, '00000055': 2, '00000057': 2, '00000058': 4, '00000060': 1, '00000061': 5, '00000062': 3, '00000063': 2, '00000064': 3, '00000066': 5, '00000068': 5, '00000069': 4, '00000070': 3, '00000071': 4, '00000072': 2, '00000074': 1, '00000075': 5, '00000078': 3, '00000079': 3, '00000080': 5, '00000081': 4, '00000082': 3, '00000083': 4, '00000084': 2, '00000085': 3, '00000086': 5, '00000088': 1, '00000089': 2, '00000090': 2, '00000092': 4, '00000093': 1, '00000095': 1, '00000096': 2, '00000097': 5, '00000098': 4, '00000099': 5}

def make_trip_folds(dataset, out, protocol=Protocol()):
    """Reuse the exact trip folds already used for LightGBM and XGBoost."""
    ids = set(dataset.observations.trip_id)
    if ids != set(REFERENCE_FOLDS):
        raise ValueError("Trip IDs differ from the original 80-trip benchmark")
    assignment = pd.DataFrame(sorted(REFERENCE_FOLDS.items()), columns=['trip_id', 'fold'])
    assignment.to_csv(Path(out) / 'fold_assignments.csv', index=False)
    return assignment.set_index('trip_id').fold


def subset_by_trips(dataset, trips):
    return Dataset(dataset.candidates.loc[dataset.candidates.trip_id.isin(trips)].copy(),
                   dataset.observations.loc[dataset.observations.trip_id.isin(trips)].copy())



# %% Common numeric matrices, fitted transformations, and fold access
from sklearn.preprocessing import StandardScaler
import joblib


def export_shared_inputs(datasets, output_dir):
    out = Path(output_dir)
    train = datasets['train']
    test = datasets['student_test']
    assert not set(train.observations.trip_id) & set(test.observations.trip_id)
    X = train.candidates[FEATURES].to_numpy(dtype=np.float32)
    Xt = test.candidates[FEATURES].to_numpy(dtype=np.float32)
    y = train.candidates.label.to_numpy(dtype=np.int8)
    yt = test.candidates.label.to_numpy(dtype=np.int8)
    assert X.shape[1] == Xt.shape[1] == 6
    assert np.isfinite(X).all() and np.isfinite(Xt).all()
    # These deterministic geometric features should always be finite. Fail on
    # an upstream error instead of silently dropping candidates or inventing data.
    trip_folds = make_trip_folds(train, out)
    row_folds = train.candidates.trip_id.map(trip_folds).to_numpy(dtype=np.int8)
    groups = train.candidates.trip_id.to_numpy(dtype='U')
    scaler = StandardScaler().fit(X)
    joblib.dump(scaler, out / 'final_scaler.joblib')
    np.savez_compressed(out / 'shared_inputs.npz',
        X_train_raw=X, X_train_scaled=scaler.transform(X).astype(np.float32),
        y_train=y, X_test_raw=Xt, X_test_scaled=scaler.transform(Xt).astype(np.float32),
        y_test=yt, features=np.asarray(FEATURES, dtype='U'),
        train_trip_id=groups, test_trip_id=test.candidates.trip_id.to_numpy(dtype='U'),
        train_observation_id=train.candidates.observation_id.to_numpy(),
        test_observation_id=test.candidates.observation_id.to_numpy(),
        train_arc_id=train.candidates.arc_id.to_numpy(),
        test_arc_id=test.candidates.arc_id.to_numpy(), fold_id=row_folds)
    for fold in range(1, 6):
        fit_idx = np.flatnonzero(row_folds != fold)
        valid_idx = np.flatnonzero(row_folds == fold)
        assert not set(groups[fit_idx]) & set(groups[valid_idx])
        assert set(np.unique(y[fit_idx])) == {0, 1}
        fold_scaler = StandardScaler().fit(X[fit_idx])
        joblib.dump(fold_scaler, out / f'fold_{fold}_scaler.joblib')
        np.savez_compressed(out / f'fold_{fold}_indices.npz',
                            fit_idx=fit_idx, valid_idx=valid_idx)
    versions = {p: importlib.metadata.version(p) for p in
                ['numpy', 'pandas', 'scipy', 'scikit-learn', 'shapely', 'pyproj', 'joblib']}
    write_json(out / 'shared_protocol.json', dict(
        features=FEATURES, versions=versions, positive_class=1,
        preprocessing='StandardScaler; fitted separately on each training partition',
        cv='Original five fixed trip folds; never split candidate rows randomly',
        labels='Geometric weak labels derived independently from supplied routes',
        missing_heading='90 degrees; first-observation displacement is zero',
        class_balance='Original candidate distribution retained; no resampling',
        test_policy='Use student_test once after model/hyperparameter selection',
        train_candidates=len(X), test_candidates=len(Xt)))
    return out / 'shared_inputs.npz'


def load_fold(output_dir, fold, scaled=True):
    """Same rows, labels, order and train-fitted transform for every model."""
    if fold not in range(1, 6):
        raise ValueError('fold must be 1..5')
    out = Path(output_dir)
    with np.load(out / 'shared_inputs.npz', allow_pickle=False) as data:
        X, y = data['X_train_raw'], data['y_train']
    with np.load(out / f'fold_{fold}_indices.npz') as indices:
        fit_idx, valid_idx = indices['fit_idx'], indices['valid_idx']
    X_fit, X_valid = X[fit_idx], X[valid_idx]
    if scaled:
        scaler = joblib.load(out / f'fold_{fold}_scaler.joblib')
        X_fit = scaler.transform(X_fit).astype(np.float32)
        X_valid = scaler.transform(X_valid).astype(np.float32)
    return X_fit, X_valid, y[fit_idx], y[valid_idx], fit_idx, valid_idx


## 3. Build exactly six features and export reusable inputs


In [4]:
datasets = prepare_datasets(DATA_ROOT, OUTPUT_DIR, protocol=Protocol())
shared_path = export_shared_inputs(datasets, OUTPUT_DIR)
# The next cell saves the reusable loader and scoring module with the dataset.
with np.load(shared_path, allow_pickle=False) as arrays:
    print('Features:', arrays['features'].tolist())
    print('Training:', arrays['X_train_scaled'].shape, 'Test:', arrays['X_test_scaled'].shape)
    print('Positive-class fraction:', arrays['y_train'].mean())
display(pd.read_json(OUTPUT_DIR / 'feature_schema.json'))
print('Saved:', shared_path)


Verifying content hashes of the 360 permitted input files...
train: prepared 1/80 trips (1,315 observations)
train: prepared 10/80 trips (23,178 observations)
train: prepared 20/80 trips (50,834 observations)
train: prepared 30/80 trips (75,365 observations)
train: prepared 40/80 trips (92,631 observations)
train: prepared 50/80 trips (122,990 observations)
train: prepared 60/80 trips (144,205 observations)
train: prepared 70/80 trips (182,206 observations)
train: prepared 80/80 trips (208,417 observations)
student_test: prepared 1/10 trips (2,503 observations)
student_test: prepared 10/10 trips (19,160 observations)
Features: ['distance_m', 'distance_rank', 'heading_difference_deg', 'projection_fraction', 'segment_length_m', 'displacement_m']
Training: (1241756, 6) Test: (125525, 6)
Positive-class fraction: 0.1492636234493733


,name,definition,rationale
0,distance_m,Shortest projected GPS-to-segment distance (me...,Measures spatial compatibility.
1,distance_rank,1-based distance rank; equal distances break b...,Expresses relative proximity among candidates.
2,heading_difference_deg,Absolute wrapped movement/segment heading diff...,Distinguishes opposite travel directions.
3,projection_fraction,Clipped fraction along segment of the perpendi...,Identifies endpoint and interior projections.
4,segment_length_m,Length of the directed straight segment (metres).,Distinguishes short connectors from long roads.
5,displacement_m,Projected displacement from the preceding reta...,Describes movement magnitude and GPS noise.


Saved: /content/drive/MyDrive/output/Shared6/shared_inputs.npz


In [5]:
(OUTPUT_DIR / "pathsnap_shared6.py").write_text('# %% Imports and fixed experimental protocol\nfrom dataclasses import asdict, dataclass\nfrom hashlib import sha256\nfrom pathlib import Path\nfrom time import perf_counter\nimport argparse\nimport importlib.metadata\nimport json\nimport os\nimport platform\n\nimport numpy as np\nimport pandas as pd\nimport shapely\nfrom pyproj import Geod, Transformer\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.csgraph import connected_components\nfrom shapely import STRtree\nfrom sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,\n                             log_loss, precision_score, recall_score)\nfrom sklearn.model_selection import GroupKFold\n\nRANDOM_STATE = 42\nnp.random.seed(RANDOM_STATE)\n\n\n@dataclass(frozen=True)\nclass Protocol:\n    schema_version: str = "shared-six-v1"\n    candidate_radius_m: float = 75.0\n    max_candidates: int = 8\n    heading_min_displacement_m: float = 3.0\n    heading_max_gap_s: float = 30.0\n    n_splits: int = 5\n    random_state: int = RANDOM_STATE\n\n\n# Exactly six ordered numeric model inputs. IDs and labels stay outside X.\n# First observations have displacement=0 and heading_difference_deg=90.\n# No heading-validity indicator is included in this selected feature set.\nFEATURE_INFO = [(\'distance_m\', \'Shortest projected GPS-to-segment distance (metres).\', \'Measures spatial compatibility.\'), (\'distance_rank\', \'1-based distance rank; equal distances break by arc ID.\', \'Expresses relative proximity among candidates.\'), (\'heading_difference_deg\', \'Absolute wrapped movement/segment heading difference, 0..180; 90 if unavailable.\', \'Distinguishes opposite travel directions.\'), (\'projection_fraction\', \'Clipped fraction along segment of the perpendicular GPS projection, 0..1.\', \'Identifies endpoint and interior projections.\'), (\'segment_length_m\', \'Length of the directed straight segment (metres).\', \'Distinguishes short connectors from long roads.\'), (\'displacement_m\', \'Projected displacement from the preceding retained GPS observation; first=0.\', \'Describes movement magnitude and GPS noise.\')]\nFEATURES = [f[0] for f in FEATURE_INFO]\nMETRICS = ["Accuracy", "Precision", "Recall", "F1", "Log Loss", "Top-1 Accuracy",\n           "Candidate Miss Rate", "Train Time", "Inference Time"]\nCOMPARISON_COLUMNS = ["Model"] + METRICS\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef resolve_data_root(path):\n    """Accept either input/, input/Student folder/, or an equivalent Colab path."""\n    path = Path(path).expanduser().resolve()\n    for root in (path, path / "Student folder"):\n        if (root / "train").is_dir() and (root / "student_test").is_dir():\n            return root\n    raise FileNotFoundError(f"Expected train/ and student_test/ under {path} or its Student folder/ child")\n\n\ndef discover_trips(root):\n    """Allowlist exactly two splits; never open or enumerate evaluator files."""\n    splits = {}\n    for split, expected in (("train", 80), ("student_test", 10)):\n        trips = sorted(p for p in (root / split).iterdir() if p.is_dir())\n        if len(trips) != expected:\n            raise ValueError(f"{split}: expected {expected} trips, found {len(trips)}")\n        for trip in trips:\n            for ext in ("track", "nodes", "arcs", "route"):\n                path = trip / f"{trip.name}.{ext}"\n                if not path.is_file() or path.stat().st_size == 0:\n                    raise FileNotFoundError(f"Required nonempty supervised input missing: {path}")\n        splits[split] = trips\n    if {p.name for p in splits["train"]} & {p.name for p in splits["student_test"]}:\n        raise ValueError("Trip IDs overlap between train and student_test")\n    return splits\n\n\ndef read_numeric(path, columns, integer=False):\n    """Parse headerless whitespace, reject wrong columns/non-numeric/invalid IDs."""\n    try:\n        a = np.loadtxt(path, dtype=np.float64, ndmin=2)\n    except (ValueError, OSError) as exc:\n        raise ValueError(f"Cannot parse {path}: {exc}") from exc\n    if not len(a) or a.shape[1] != columns:\n        raise ValueError(f"{path}: expected {columns} columns, got shape {a.shape}")\n    if integer:\n        if not np.isfinite(a).all() or not np.equal(a, np.floor(a)).all():\n            raise ValueError(f"{path}: expected finite integer IDs")\n        a = a.astype(np.int64)\n    return a\n\n\ndef coordinates_valid(a):\n    return np.isfinite(a[:, :2]).all(axis=1) & (np.abs(a[:, 0]) <= 180) & (np.abs(a[:, 1]) <= 90)\n\n\ndef preprocess_track(raw):\n    """Stable timestamp sort, retain original row IDs, drop invalid/duplicate time rows."""\n    good = coordinates_valid(raw) & np.isfinite(raw[:, 2])\n    frame = pd.DataFrame(raw[good], columns=["longitude", "latitude", "timestamp"])\n    frame["raw_row"] = np.flatnonzero(good)\n    original_order = frame.raw_row.to_numpy().copy()\n    frame = frame.sort_values("timestamp", kind="stable")\n    reordered = int(np.sum(original_order != frame.raw_row.to_numpy()))\n    duplicates = int(frame.timestamp.duplicated().sum())\n    frame = frame.drop_duplicates("timestamp", keep="first").reset_index(drop=True)\n    if len(frame) < 2:\n        raise ValueError("Trip has fewer than two valid, distinct-time GPS observations")\n    assert (np.diff(frame.timestamp) > 0).all()\n    return frame, dict(raw_observations=len(raw), invalid_gps_dropped=int((~good).sum()),\n                       duplicate_times_dropped=duplicates, reordered_rows=reordered)\n\n\n@dataclass\nclass Network:\n    nodes_xy: np.ndarray\n    arcs: np.ndarray\n    lines: np.ndarray\n    lengths: np.ndarray\n    headings: np.ndarray\n    active_ids: np.ndarray\n    tree: STRtree\n    out_degree: np.ndarray\n    in_degree: np.ndarray\n\n\ndef make_network(nodes_xy, arcs):\n    endpoints = nodes_xy[arcs]\n    delta = endpoints[:, 1] - endpoints[:, 0]\n    lengths = np.linalg.norm(delta, axis=1)\n    active = np.flatnonzero(lengths > 1e-6)\n    if not len(active):\n        raise ValueError("No nondegenerate road segments")\n    lines = shapely.linestrings(endpoints)\n    return Network(nodes_xy, arcs, lines, lengths, np.arctan2(delta[:, 1], delta[:, 0]),\n                   active, STRtree(lines[active]),\n                   np.bincount(arcs[active, 0], minlength=len(nodes_xy)),\n                   np.bincount(arcs[active, 1], minlength=len(nodes_xy)))\n\n\ndef load_trip(trip):\n    paths = {ext: trip / f"{trip.name}.{ext}" for ext in ("track", "nodes", "arcs", "route")}\n    track, quality = preprocess_track(read_numeric(paths["track"], 3))\n    nodes = read_numeric(paths["nodes"], 2)\n    arcs = read_numeric(paths["arcs"], 2, integer=True)\n    route = read_numeric(paths["route"], 1, integer=True).ravel()\n    if not coordinates_valid(nodes).all():\n        raise ValueError(f"{trip}: nonfinite/out-of-range node coordinates")\n    if arcs.min() < 0 or arcs.max() >= len(nodes):\n        raise ValueError(f"{trip}: arc endpoint outside zero-based node index")\n    if route.min() < 0 or route.max() >= len(arcs):\n        raise ValueError(f"{trip}: route ID outside zero-based arc index")\n    # A separate local UTM zone/hemisphere for each trip. No route labels or\n    # fitted statistics are used by this deterministic coordinate conversion.\n    lon, lat = track[["longitude", "latitude"]].median().to_numpy()\n    if not -80 <= lat <= 84:\n        raise ValueError(f"{trip}: outside supported UTM latitude range")\n    zone = min(60, max(1, int((lon + 180) // 6) + 1))\n    epsg = (32600 if lat >= 0 else 32700) + zone\n    transform = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)\n    nodes_xy = np.column_stack(transform.transform(nodes[:, 0], nodes[:, 1]))\n    track_xy = np.column_stack(transform.transform(track.longitude.to_numpy(), track.latitude.to_numpy()))\n    if not np.isfinite(nodes_xy).all() or not np.isfinite(track_xy).all():\n        raise ValueError(f"{trip}: invalid projection output")\n    net = make_network(nodes_xy, arcs)\n    adj = coo_matrix((np.ones(len(arcs), dtype=np.int8), (arcs[:, 0], arcs[:, 1])),\n                     shape=(len(nodes), len(nodes))).tocsr()\n    components = connected_components(adj, directed=False, return_labels=False)\n    sample = net.active_ids[np.linspace(0, len(net.active_ids) - 1, min(200, len(net.active_ids)), dtype=int)]\n    a, b = nodes[arcs[sample, 0]], nodes[arcs[sample, 1]]\n    _, _, geod_lengths = Geod(ellps="WGS84").inv(a[:, 0], a[:, 1], b[:, 0], b[:, 1])\n    relative_errors = np.abs(net.lengths[sample] / geod_lengths - 1)\n    if relative_errors.max() > 0.01:\n        raise ValueError(f"{trip}: sampled UTM/geodesic length discrepancy exceeds 1%; review CRS")\n    dt = np.diff(track.timestamp)\n    quality.update(trip_id=trip.name, observations=len(track), nodes=len(nodes), arcs=len(arcs),\n                   route_entries=len(route), route_repeated_arcs=int(len(route) - len(np.unique(route))),\n                   route_disconnects=int((arcs[route[:-1], 1] != arcs[route[1:], 0]).sum()),\n                   zero_length_arcs=int((net.lengths <= 1e-6).sum()),\n                   weak_components=int(components), isolated_nodes=int((net.in_degree + net.out_degree == 0).sum()),\n                   epsg=epsg, projection_max_relative_error=float(relative_errors.max()),\n                   longitude_min=float(track.longitude.min()), longitude_max=float(track.longitude.max()),\n                   latitude_min=float(track.latitude.min()), latitude_max=float(track.latitude.max()),\n                   timestamp_min=float(track.timestamp.min()), timestamp_max=float(track.timestamp.max()),\n                   dt_min=float(dt.min()), dt_max=float(dt.max()), steps_1s=int((dt == 1).sum()),\n                   gaps_over_30s=int((dt > 30).sum()))\n    return track, track_xy, net, route, quality\n\n\n# %% Label-free spatial candidates and common features\ndef movement_features(track, xy, protocol):\n    delta = np.diff(xy, axis=0, prepend=xy[:1])\n    displacement = np.linalg.norm(delta, axis=1)\n    dt = np.diff(track.timestamp, prepend=track.timestamp.iloc[0])\n    valid = (displacement >= protocol.heading_min_displacement_m) & (dt > 0) & (dt <= protocol.heading_max_gap_s)\n    speed = np.divide(displacement, dt, out=np.zeros_like(dt), where=dt > 0)\n    return dt, displacement, speed, np.arctan2(delta[:, 1], delta[:, 0]), valid\n\n\ndef heading_difference(movement, segment):\n    return np.rad2deg(np.abs((movement - segment + np.pi) % (2 * np.pi) - np.pi))\n\n\ndef generate_candidates(track, xy, net, protocol=Protocol()):\n    """R-tree radius search, exact distances, nearest k. Never receives .route.\n\n    No forced reference injection and no label-dependent radius expansion.\n    Observations without a segment within the radius remain in the ledger.\n    """\n    points = shapely.points(xy)\n    pair = net.tree.query(points, predicate="dwithin", distance=protocol.candidate_radius_m)\n    obs_id = pair[0].astype(np.int32)\n    arc_id = net.active_ids[pair[1]].astype(np.int32)\n    distances = shapely.distance(points[obs_id], net.lines[arc_id])\n    # Stable deterministic ordering, including reversed arcs with equal geometry.\n    order = np.lexsort((arc_id, distances, obs_id))\n    obs_id, arc_id, distances = obs_id[order], arc_id[order], distances[order]\n    starts = np.r_[0, np.flatnonzero(np.diff(obs_id)) + 1] if len(obs_id) else np.array([], dtype=int)\n    rank = np.arange(len(obs_id)) - np.repeat(starts, np.diff(np.r_[starts, len(obs_id)])) + 1\n    keep = rank <= protocol.max_candidates\n    obs_id, arc_id, distances, rank = obs_id[keep], arc_id[keep], distances[keep], rank[keep]\n    dt, displacement, speed, heading, valid = movement_features(track, xy, protocol)\n    endpoints = net.nodes_xy[net.arcs[arc_id]]\n    vector = endpoints[:, 1] - endpoints[:, 0]\n    fraction = np.einsum("ij,ij->i", xy[obs_id] - endpoints[:, 0], vector) / net.lengths[arc_id] ** 2\n    hd = heading_difference(heading[obs_id], net.headings[arc_id])\n    hd[~valid[obs_id]] = 90.0\n    values = np.column_stack([\n        distances, rank, hd, np.clip(fraction, 0, 1),\n        net.lengths[arc_id], displacement[obs_id],\n    ]).astype(np.float32)\n    candidates = pd.DataFrame(values, columns=FEATURES)\n    candidates.insert(0, "arc_id", arc_id)\n    candidates.insert(0, "observation_id", obs_id)\n    assert np.isfinite(values).all()\n    assert not candidates.duplicated(["observation_id", "arc_id"]).any()\n    return candidates\n\n\n# %% Independent construction of geometric reference labels\ndef derive_reference_labels(track, xy, net, route, protocol=Protocol()):\n    """Weak observation labels from the supplied route, independently of candidates.\n\n    Choose the nearest nonzero directed segment anywhere on the reference route.\n    Exact geometric ties: heading agreement (if valid), then proximity to a\n    rough progress estimate from cumulative observed displacement, then route\n    position. Chronology informs ties only: this is not a sequence decoder.\n    Route breaks are recorded, never filled with invented connecting segments.\n    """\n    positions = np.flatnonzero(net.lengths[route] > 1e-6)\n    if not len(positions):\n        raise ValueError("Reference route has no nondegenerate segment")\n    ids = route[positions]\n    points = shapely.points(xy)\n    pairs, distance = STRtree(net.lines[ids]).query_nearest(points, all_matches=True, return_distance=True)\n    obs_id, route_pos = pairs[0], positions[pairs[1]]\n    arc_id = route[route_pos]\n    _, displacement, _, heading, valid = movement_features(track, xy, protocol)\n    prefix = np.r_[0.0, np.cumsum(net.lengths[route])]\n    observed_progress = np.cumsum(displacement)\n    expected_progress = observed_progress / max(observed_progress[-1], 1e-6) * prefix[-1]\n    projected = shapely.line_locate_point(net.lines[arc_id], points[obs_id])\n    progress = prefix[route_pos] + projected\n    hd = heading_difference(heading[obs_id], net.headings[arc_id])\n    hd[~valid[obs_id]] = 0\n    order = np.lexsort((route_pos, np.abs(progress - expected_progress[obs_id]), hd, obs_id))\n    ordered_obs = obs_id[order]\n    selected = order[np.r_[True, np.diff(ordered_obs) != 0]]\n    assert np.array_equal(obs_id[selected], np.arange(len(track)))\n    matches = pd.DataFrame({"obs": obs_id, "arc": arc_id}).drop_duplicates()\n    ambiguous = np.bincount(matches.obs.to_numpy(), minlength=len(track)) > 1\n    return pd.DataFrame(dict(\n        reference_arc_id=arc_id[selected].astype(np.int32),\n        reference_distance_m=distance[selected], reference_route_position=route_pos[selected],\n        reference_progress_m=progress[selected], reference_tie=ambiguous,\n    ))\n\n\ndef attach_labels(candidates, observations):\n    """Keep candidate misses, including observations with zero candidate rows."""\n    candidates = candidates.copy()\n    obs_idx = candidates.observation_id.to_numpy()\n    candidates["label"] = (candidates.arc_id.to_numpy() == observations.reference_arc_id.to_numpy()[obs_idx]).astype(np.int8)\n    counts = np.bincount(obs_idx, minlength=len(observations))\n    positives = np.bincount(obs_idx, weights=candidates.label.to_numpy(), minlength=len(observations))\n    observations = observations.copy()\n    observations["candidate_count"] = counts\n    observations["reference_present"] = positives > 0\n    assert (positives <= 1).all()\n    assert int(candidates.label.sum()) == int(observations.reference_present.sum())\n    return candidates, observations\n\n\n@dataclass\nclass Dataset:\n    candidates: pd.DataFrame\n    observations: pd.DataFrame\n\n\ndef input_manifest(splits):\n    """Content hashes make caches invalid when any permitted source file changes."""\n    files = []\n    for split, trips in splits.items():\n        for trip in trips:\n            for ext in ("track", "nodes", "arcs", "route"):\n                path = trip / f"{trip.name}.{ext}"\n                files.append(dict(split=split, trip_id=trip.name, file=path.name,\n                                  bytes=path.stat().st_size, sha256=sha256(path.read_bytes()).hexdigest()))\n    return files\n\n\ndef build_dataset(trips, split, protocol, out):\n    candidate_frames, observation_frames, inventory = [], [], []\n    offset = 0\n    for number, trip in enumerate(trips, 1):\n        start = perf_counter()\n        track, xy, net, route, quality = load_trip(trip)\n        candidates = generate_candidates(track, xy, net, protocol)\n        # Reference information enters here, after all features are finished.\n        labels = derive_reference_labels(track, xy, net, route, protocol)\n        observations = pd.concat([track, labels], axis=1)\n        candidates, observations = attach_labels(candidates, observations)\n        observations.insert(0, "observation_id", np.arange(len(track)) + offset)\n        observations.insert(0, "trip_id", trip.name)\n        candidates.observation_id += offset\n        candidates.insert(0, "trip_id", trip.name)\n        quality.update(split=split, candidate_rows=len(candidates),\n                       positive_candidates=int(candidates.label.sum()),\n                       candidate_misses=int((~observations.reference_present).sum()),\n                       zero_candidate_observations=int((observations.candidate_count == 0).sum()),\n                       reference_ties=int(observations.reference_tie.sum()),\n                       reference_over_radius=int((observations.reference_distance_m > protocol.candidate_radius_m).sum()),\n                       reference_progress_backsteps=int((np.diff(observations.reference_progress_m) < -1).sum()),\n                       preprocessing_seconds=perf_counter() - start)\n        inventory.append(quality)\n        candidate_frames.append(candidates)\n        observation_frames.append(observations)\n        offset += len(track)\n        if number == 1 or number % 10 == 0 or number == len(trips):\n            print(f"{split}: prepared {number}/{len(trips)} trips ({offset:,} observations)", flush=True)\n    dataset = Dataset(pd.concat(candidate_frames, ignore_index=True), pd.concat(observation_frames, ignore_index=True))\n    save_dataset(dataset, out, split)\n    pd.DataFrame(inventory).to_csv(out / f"{split}_inventory.csv", index=False)\n    return dataset\n\n\ndef save_dataset(dataset, out, split):\n    c = dataset.candidates\n    np.savez_compressed(out / f"{split}_features.npz", X=c[FEATURES].to_numpy(np.float32),\n                        label=c.label.to_numpy(np.int8), trip_id=c.trip_id.to_numpy(dtype="U"),\n                        observation_id=c.observation_id.to_numpy(np.int32), arc_id=c.arc_id.to_numpy(np.int32))\n    c.to_csv(out / f"{split}_candidates.csv.gz", index=False, compression="gzip")\n    dataset.observations.to_csv(out / f"{split}_observations.csv.gz", index=False, compression="gzip")\n\n\ndef load_dataset(out, split):\n    with np.load(out / f"{split}_features.npz", allow_pickle=False) as cache:\n        c = pd.DataFrame(cache["X"], columns=FEATURES)\n        for key in ("trip_id", "observation_id", "arc_id", "label"):\n            c[key] = cache[key]\n    obs = pd.read_csv(out / f"{split}_observations.csv.gz", dtype={"trip_id": str})\n    return Dataset(c, obs)\n\n\ndef prepare_datasets(data_root, output_dir, protocol=Protocol(), rebuild=False):\n    root = resolve_data_root(data_root)\n    out = Path(output_dir).expanduser().resolve()\n    if out == root or root in out.parents:\n        raise ValueError("Output directory must be outside the raw dataset")\n    out.mkdir(parents=True, exist_ok=True)\n    splits = discover_trips(root)\n    print("Verifying content hashes of the 360 permitted input files...", flush=True)\n    manifest = input_manifest(splits)\n    signature = {"protocol": asdict(protocol), "features": FEATURES, "inputs": manifest,\n                 "spatial_versions": {k: importlib.metadata.version(k) for k in ("numpy", "shapely", "pyproj")}}\n    # This fixed schema version must be incremented for changes to feature/label\n    # semantics. Protocol values, file contents and spatial versions are hashed.\n    digest = sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()\n    cache_path = out / "cache_manifest.json"\n    required = [out / f"{s}_{suffix}" for s in splits for suffix in\n                ("features.npz", "observations.csv.gz", "inventory.csv", "candidates.csv.gz")]\n    valid_cache = (not rebuild and cache_path.is_file() and all(p.is_file() for p in required)\n                   and json.loads(cache_path.read_text(encoding="utf-8"))["digest"] == digest)\n    if valid_cache:\n        print("Reusing verified common feature/label caches.", flush=True)\n        datasets = {s: load_dataset(out, s) for s in splits}\n    else:\n        datasets = {s: build_dataset(trips, s, protocol, out) for s, trips in splits.items()}\n        write_json(cache_path, dict(digest=digest, **signature))\n    write_json(out / "feature_schema.json", [dict(name=n, definition=d, rationale=r) for n, d, r in FEATURE_INFO])\n    pd.concat([pd.read_csv(out / f"{s}_inventory.csv", dtype={"trip_id": str}) for s in splits]).to_csv(out / "dataset_inventory.csv", index=False)\n    return datasets\n\n\n# %% Shared model-independent metrics and trip-aware folds\ndef evaluate_predictions(dataset, probabilities, train_seconds=0.0, inference_seconds=0.0):\n    """Binary metrics on generated rows; Top-1 on ALL labeled observations.\n\n    Miss rate is a fraction (multiply by 100 for percent). Missing-reference and\n    zero-candidate observations count as pipeline Top-1 failures. A separate\n    covered-only score isolates the classifier from candidate-generation misses.\n    All ranking ties prefer the existing distance-rank/arc-ID order.\n    """\n    c, obs = dataset.candidates, dataset.observations\n    p = np.asarray(probabilities, dtype=np.float64)\n    if p.shape != (len(c),) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Expected one finite probability in [0, 1] per candidate")\n    if c.empty:\n        raise ValueError("No candidate rows in evaluation partition; classification metrics undefined")\n    y, pred = c.label.to_numpy(), (p >= 0.5).astype(np.int8)\n    scored = c[["observation_id", "arc_id", "distance_rank"]].copy()\n    scored["probability"] = p\n    best = scored.sort_values(["observation_id", "probability", "distance_rank", "arc_id"],\n                              ascending=[True, False, True, True], kind="stable").drop_duplicates("observation_id")\n    per_obs = obs.merge(best[["observation_id", "arc_id", "probability"]], on="observation_id", how="left", validate="one_to_one")\n    per_obs.rename(columns={"arc_id": "predicted_arc_id", "probability": "predicted_probability"}, inplace=True)\n    per_obs["top1_correct"] = per_obs.predicted_arc_id == per_obs.reference_arc_id\n    covered = per_obs.reference_present.astype(bool)\n    matrix = confusion_matrix(y, pred, labels=[0, 1])\n    scores = dict(zip(METRICS, [\n        float(accuracy_score(y, pred)), float(precision_score(y, pred, zero_division=0)),\n        float(recall_score(y, pred, zero_division=0)), float(f1_score(y, pred, zero_division=0)),\n        float(log_loss(y, np.column_stack([1 - p, p]), labels=[0, 1])),\n        float(per_obs.top1_correct.mean()), float((~covered).mean()),\n        float(train_seconds), float(inference_seconds),\n    ]))\n    # The supplied final-report template also asks for Macro F1; keep the\n    # requested common F1 explicitly positive-class, and supply macro separately.\n    scores.update({"Macro F1": float(f1_score(y, pred, average="macro", labels=[0, 1], zero_division=0)),\n                   "Top-1 Accuracy (covered)": float(per_obs.loc[covered, "top1_correct"].mean()) if covered.any() else None,\n                   "Observations": len(obs), "Candidates": len(c),\n                   "Candidate Miss Count": int((~covered).sum()),\n                   "Zero Candidate Observations": int((per_obs.candidate_count == 0).sum())})\n    return scores, matrix, per_obs\n\n\nREFERENCE_FOLDS = {\'00000001\': 4, \'00000002\': 5, \'00000003\': 3, \'00000004\': 4, \'00000005\': 4, \'00000006\': 2, \'00000008\': 1, \'00000010\': 1, \'00000011\': 5, \'00000012\': 3, \'00000013\': 4, \'00000014\': 5, \'00000016\': 2, \'00000018\': 3, \'00000019\': 4, \'00000020\': 1, \'00000021\': 3, \'00000022\': 5, \'00000024\': 2, \'00000025\': 5, \'00000026\': 1, \'00000027\': 1, \'00000028\': 4, \'00000029\': 5, \'00000031\': 2, \'00000032\': 3, \'00000033\': 4, \'00000034\': 1, \'00000035\': 2, \'00000036\': 3, \'00000037\': 2, \'00000038\': 4, \'00000039\': 2, \'00000041\': 3, \'00000042\': 5, \'00000043\': 2, \'00000044\': 1, \'00000045\': 4, \'00000046\': 1, \'00000047\': 3, \'00000048\': 1, \'00000050\': 3, \'00000051\': 5, \'00000053\': 5, \'00000054\': 1, \'00000055\': 2, \'00000057\': 2, \'00000058\': 4, \'00000060\': 1, \'00000061\': 5, \'00000062\': 3, \'00000063\': 2, \'00000064\': 3, \'00000066\': 5, \'00000068\': 5, \'00000069\': 4, \'00000070\': 3, \'00000071\': 4, \'00000072\': 2, \'00000074\': 1, \'00000075\': 5, \'00000078\': 3, \'00000079\': 3, \'00000080\': 5, \'00000081\': 4, \'00000082\': 3, \'00000083\': 4, \'00000084\': 2, \'00000085\': 3, \'00000086\': 5, \'00000088\': 1, \'00000089\': 2, \'00000090\': 2, \'00000092\': 4, \'00000093\': 1, \'00000095\': 1, \'00000096\': 2, \'00000097\': 5, \'00000098\': 4, \'00000099\': 5}\n\ndef make_trip_folds(dataset, out, protocol=Protocol()):\n    """Reuse the exact trip folds already used for LightGBM and XGBoost."""\n    ids = set(dataset.observations.trip_id)\n    if ids != set(REFERENCE_FOLDS):\n        raise ValueError("Trip IDs differ from the original 80-trip benchmark")\n    assignment = pd.DataFrame(sorted(REFERENCE_FOLDS.items()), columns=[\'trip_id\', \'fold\'])\n    assignment.to_csv(Path(out) / \'fold_assignments.csv\', index=False)\n    return assignment.set_index(\'trip_id\').fold\n\n\ndef subset_by_trips(dataset, trips):\n    return Dataset(dataset.candidates.loc[dataset.candidates.trip_id.isin(trips)].copy(),\n                   dataset.observations.loc[dataset.observations.trip_id.isin(trips)].copy())\n\n\n\n# %% Common numeric matrices, fitted transformations, and fold access\nfrom sklearn.preprocessing import StandardScaler\nimport joblib\n\n\ndef export_shared_inputs(datasets, output_dir):\n    out = Path(output_dir)\n    train = datasets[\'train\']\n    test = datasets[\'student_test\']\n    assert not set(train.observations.trip_id) & set(test.observations.trip_id)\n    X = train.candidates[FEATURES].to_numpy(dtype=np.float32)\n    Xt = test.candidates[FEATURES].to_numpy(dtype=np.float32)\n    y = train.candidates.label.to_numpy(dtype=np.int8)\n    yt = test.candidates.label.to_numpy(dtype=np.int8)\n    assert X.shape[1] == Xt.shape[1] == 6\n    assert np.isfinite(X).all() and np.isfinite(Xt).all()\n    # These deterministic geometric features should always be finite. Fail on\n    # an upstream error instead of silently dropping candidates or inventing data.\n    trip_folds = make_trip_folds(train, out)\n    row_folds = train.candidates.trip_id.map(trip_folds).to_numpy(dtype=np.int8)\n    groups = train.candidates.trip_id.to_numpy(dtype=\'U\')\n    scaler = StandardScaler().fit(X)\n    joblib.dump(scaler, out / \'final_scaler.joblib\')\n    np.savez_compressed(out / \'shared_inputs.npz\',\n        X_train_raw=X, X_train_scaled=scaler.transform(X).astype(np.float32),\n        y_train=y, X_test_raw=Xt, X_test_scaled=scaler.transform(Xt).astype(np.float32),\n        y_test=yt, features=np.asarray(FEATURES, dtype=\'U\'),\n        train_trip_id=groups, test_trip_id=test.candidates.trip_id.to_numpy(dtype=\'U\'),\n        train_observation_id=train.candidates.observation_id.to_numpy(),\n        test_observation_id=test.candidates.observation_id.to_numpy(),\n        train_arc_id=train.candidates.arc_id.to_numpy(),\n        test_arc_id=test.candidates.arc_id.to_numpy(), fold_id=row_folds)\n    for fold in range(1, 6):\n        fit_idx = np.flatnonzero(row_folds != fold)\n        valid_idx = np.flatnonzero(row_folds == fold)\n        assert not set(groups[fit_idx]) & set(groups[valid_idx])\n        assert set(np.unique(y[fit_idx])) == {0, 1}\n        fold_scaler = StandardScaler().fit(X[fit_idx])\n        joblib.dump(fold_scaler, out / f\'fold_{fold}_scaler.joblib\')\n        np.savez_compressed(out / f\'fold_{fold}_indices.npz\',\n                            fit_idx=fit_idx, valid_idx=valid_idx)\n    versions = {p: importlib.metadata.version(p) for p in\n                [\'numpy\', \'pandas\', \'scipy\', \'scikit-learn\', \'shapely\', \'pyproj\', \'joblib\']}\n    write_json(out / \'shared_protocol.json\', dict(\n        features=FEATURES, versions=versions, positive_class=1,\n        preprocessing=\'StandardScaler; fitted separately on each training partition\',\n        cv=\'Original five fixed trip folds; never split candidate rows randomly\',\n        labels=\'Geometric weak labels derived independently from supplied routes\',\n        missing_heading=\'90 degrees; first-observation displacement is zero\',\n        class_balance=\'Original candidate distribution retained; no resampling\',\n        test_policy=\'Use student_test once after model/hyperparameter selection\',\n        train_candidates=len(X), test_candidates=len(Xt)))\n    return out / \'shared_inputs.npz\'\n\n\ndef load_fold(output_dir, fold, scaled=True):\n    """Same rows, labels, order and train-fitted transform for every model."""\n    if fold not in range(1, 6):\n        raise ValueError(\'fold must be 1..5\')\n    out = Path(output_dir)\n    with np.load(out / \'shared_inputs.npz\', allow_pickle=False) as data:\n        X, y = data[\'X_train_raw\'], data[\'y_train\']\n    with np.load(out / f\'fold_{fold}_indices.npz\') as indices:\n        fit_idx, valid_idx = indices[\'fit_idx\'], indices[\'valid_idx\']\n    X_fit, X_valid = X[fit_idx], X[valid_idx]\n    if scaled:\n        scaler = joblib.load(out / f\'fold_{fold}_scaler.joblib\')\n        X_fit = scaler.transform(X_fit).astype(np.float32)\n        X_valid = scaler.transform(X_valid).astype(np.float32)\n    return X_fit, X_valid, y[fit_idx], y[valid_idx], fit_idx, valid_idx\n', encoding="utf-8")


29646

## 4. Identical input for every model
For each fold, call `load_fold` once and give **the same returned arrays** to every model. Keep `scaled=True` for all five for literally identical numeric inputs, including trees. Raw inputs are also exported, but do not mix raw/scaled policies in a strict identical-input comparison.

Do not use `X_train_scaled` for cross-validation: its scaler was fitted on all 80 training trips. Use the fold loader below, which fits no validation data.

No oversampling or balancing is applied. Set seeds in every model, use the same tuning budget, and report model fit/inference times on the same hardware. Neural models can consume the same float32 arrays directly.


In [6]:
FOLD = 1
X_fit, X_valid, y_fit, y_valid, fit_idx, valid_idx = load_fold(OUTPUT_DIR, FOLD)
print('Fold', FOLD, X_fit.shape, X_valid.shape)

# Repeat for fold=1..5 and each of YOUR five models:
# model.fit(X_fit, y_fit)
# p = model.predict_proba(X_valid)[:, list(model.classes_).index(1)]
# For a neural model: p = model.predict(X_valid).reshape(-1)
# Evaluate with the shared helper, including observations with no candidates:
# trip_folds = make_trip_folds(datasets['train'], OUTPUT_DIR)
# valid_trips = trip_folds.index[trip_folds == FOLD]
# valid_data = subset_by_trips(datasets['train'], valid_trips)
# assert np.array_equal(valid_data.candidates.index.to_numpy(), valid_idx)
# scores, confusion, observation_predictions = evaluate_predictions(valid_data, p)
# print(scores)

# After selecting model settings using CV, fit final models with the same arrays:
with np.load(shared_path, allow_pickle=False) as a:
    X_train, y_train = a['X_train_scaled'], a['y_train']
    X_test, y_test = a['X_test_scaled'], a['y_test']
# final_model.fit(X_train, y_train)
# p_test = final_model.predict_proba(X_test)[:, list(final_model.classes_).index(1)]
# scores, confusion, predictions = evaluate_predictions(datasets['student_test'], p_test)


Fold 1 (1050985, 6) (190771, 6)


## 5. Download the shared bundle
The archive contains raw/scaled matrices, labels, six feature definitions, trip/observation/arc metadata, observation ledgers, original trip folds, fold indices, fitted scalers, source hashes, package versions, and a reusable Python loader.

In another notebook, unzip the bundle, install the same dependencies, then import `load_fold` from `pathsnap_shared6`. For Top-1 scoring, use `load_dataset` and `evaluate_predictions` from the same module. Candidate misses count as Top-1 failures.


In [7]:
import shutil
from google.colab import files
bundle = shutil.make_archive('/content/PathSnap_shared6', 'zip', root_dir=OUTPUT_DIR)
files.download(bundle)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>